In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import BertTokenizer, BertModel, ViTModel, ViTImageProcessor
from huggingface_hub import HfApi, hf_hub_download

import numpy as np
from PIL import Image
from tqdm import tqdm

# Data Processing

In [5]:
dataset = load_dataset("Zoe3324/flickr30k-pairs")

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

for i in range(5):
    print(train_data[2*i])

README.md:   0%|          | 0.00/657 [00:00<?, ?B/s]

data/train-00000-of-00012.parquet:   0%|          | 0.00/97.0M [00:00<?, ?B/s]

data/train-00001-of-00012.parquet:   0%|          | 0.00/96.2M [00:00<?, ?B/s]

data/train-00002-of-00012.parquet:   0%|          | 0.00/101M [00:00<?, ?B/s]

data/train-00003-of-00012.parquet:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/145000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5070 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x7F8C8F779C10>, 'caption': 'Two young guys with shaggy hair look at their hands while hanging out in the yard.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x7F8C8D9E80B0>, 'caption': 'Two men in green shirts are standing in a yard.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=333x500 at 0x7F8C8D9E8AD0>, 'caption': 'Two friends enjoy time spent together.', 'split': 'train', 'img_id': '0', 'filename': '1000092795.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x374 at 0x7F8C8D9E81A0>, 'caption': 'Workers look down from up above on a piece of equipment.', 'split': 'train', 'img_id': '1', 'filename': '10002456.jpg'}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x374 at 0x7F8C8D9EA990>, 'c

In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

# The Model

In [8]:
class CrossAttention(nn.Module):
    def __init__(self, dim, num_heads=8, dropout=0.1, ffn_mult=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * ffn_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * ffn_mult, dim),
            nn.Dropout(dropout),
        )

    def forward(self, q, k, v):
        attn_out, _ = self.attn(q, k, v, need_weights=False)
        x = self.norm1(q + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x

In [9]:
class TwoStageModel(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.text_encoder = BertModel.from_pretrained("bert-base-uncased")
        self.image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224")

        dim = 768

        # Stage 1: projection heads for contrastive retrieval
        self.image_proj = nn.Linear(dim, embed_dim)
        self.text_proj = nn.Linear(dim, embed_dim)

        # Stage 2: cross-attention re-ranker
        self.cross_attn = CrossAttention(dim)
        self.match_head = nn.Sequential(
            nn.Linear(dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )

        self.temperature = nn.Parameter(torch.tensor(0.07))

    # Note: B=batch size, P=number of image patches, T=number of text tokens, D=hidden dimension(embedding size)
    def encode_image(self, pixel_values):
        """Encode images → (embedding for retrieval, token features for re-ranking)."""
        image_out = self.image_encoder(pixel_values=pixel_values).last_hidden_state  # (B, P, D)
        image_emb = F.normalize(self.image_proj(image_out[:, 0]), dim=-1)  # CLS token → (B, embed_dim)
        return image_emb, image_out

    def encode_text(self, input_ids, attention_mask):
        """Encode text → (embedding for retrieval, token features for re-ranking)."""
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # (B, T, D)
        text_emb = F.normalize(self.text_proj(text_out[:, 0]), dim=-1)  # [CLS] token → (B, embed_dim)
        return text_emb, text_out

    def compute_itm(self, text_tokens, image_tokens, attention_mask):
        """Stage 2: cross-attention fusion → masked pooled match score."""
        fused = self.cross_attn(text_tokens, image_tokens, image_tokens)  # (B, T, D)
        token_mask = attention_mask.unsqueeze(-1).float()
        fused = (fused * token_mask).sum(dim=1) / token_mask.sum(dim=1).clamp(min=1.0)
        logits = self.match_head(fused).squeeze(-1)  # (B,)
        return logits

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

EXPERIMENT_NAME = "bs32_e2_lr3e-5_hn5"
TRAIN_BATCH_SIZE = 32
TRAIN_EPOCHS = 2
TRAIN_LR = 3e-5
HARD_NEGATIVE_TOP_K = 3
ITM_LOSS_WEIGHT = 0.5
EARLY_STOPPING_PATIENCE = 1
EVAL_NUM_SAMPLES = 1000
EVAL_TOP_K = 20

HF_REPO_ID = "Zoe3324/advanced_3"
HF_FILENAME = f"advanced_model_{EXPERIMENT_NAME}.pt"
LOCAL_WEIGHTS_PATH = f"weights/{HF_FILENAME}"
BEST_LOCAL_WEIGHTS_PATH = f"weights/best_{HF_FILENAME}"

model = TwoStageModel().to(device)

cuda


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Trainning

In [7]:
def collate_fn(batch):
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]

    text_enc = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors="pt"
    )

    image_enc = image_processor(
        images=images,
        return_tensors="pt"
    )

    return {
        "input_ids": text_enc["input_ids"],
        "attention_mask": text_enc["attention_mask"],
        "pixel_values": image_enc["pixel_values"],
    }

In [8]:
def train(model, batch_size=32, epochs=5, lr=3e-5, hard_negative_top_k=5, itm_loss_weight=0.5, early_stopping_patience=2, best_weights_path=None):
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=4)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    itm_criterion = nn.BCEWithLogitsLoss()
    best_val_loss = float("inf")
    best_epoch = -1
    patience_counter = 0

    if best_weights_path is not None:
        os.makedirs(os.path.dirname(best_weights_path), exist_ok=True)

    def build_itm_batch(image_tokens, text_tokens, text_attention_mask, coarse_scores):
        B = coarse_scores.size(0)
        pos_logits = model.compute_itm(text_tokens, image_tokens, text_attention_mask)
        pos_labels = torch.ones(B, device=device)

        if B < 2:
            return pos_logits, pos_labels

        k = min(hard_negative_top_k, B - 1)
        masked_scores = coarse_scores.detach().clone()
        masked_scores.fill_diagonal_(-1e9)

        hard_text_candidates = masked_scores.topk(k=k, dim=1).indices
        hard_image_candidates = masked_scores.T.topk(k=k, dim=1).indices
        sampled_col = torch.randint(k, (B,), device=device)
        row_idx = torch.arange(B, device=device)
        hard_text_idx = hard_text_candidates[row_idx, sampled_col]
        hard_image_idx = hard_image_candidates[row_idx, sampled_col]

        img_to_hard_text_logits = model.compute_itm(text_tokens[hard_text_idx], image_tokens, text_attention_mask[hard_text_idx])
        hard_img_to_text_logits = model.compute_itm(text_tokens, image_tokens[hard_image_idx], text_attention_mask)
        neg_labels = torch.zeros(B, device=device)

        itm_logits = torch.cat([pos_logits, img_to_hard_text_logits, hard_img_to_text_logits], dim=0)
        itm_labels = torch.cat([pos_labels, neg_labels, neg_labels], dim=0)
        return itm_logits, itm_labels

    for epoch in range(epochs):
        # Training
        model.train()
        total_train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)

            image_emb, image_tokens = model.encode_image(pixel_values)
            text_emb, text_tokens = model.encode_text(input_ids, attention_mask)

            # Stage 1 loss: symmetric contrastive on the positive batch
            B = image_emb.size(0)
            coarse_scores = image_emb @ text_emb.T
            sim = coarse_scores / model.temperature
            contrastive_labels = torch.arange(B, device=device)
            loss_i2t = F.cross_entropy(sim, contrastive_labels)
            loss_t2i = F.cross_entropy(sim.T, contrastive_labels)
            contrastive_loss = (loss_i2t + loss_t2i) / 2

            # Stage 2 loss: ITM on positives + coarse top-k hard negatives
            itm_logits, itm_labels = build_itm_batch(image_tokens, text_tokens, attention_mask, coarse_scores)
            itm_loss = itm_criterion(itm_logits, itm_labels)

            loss = contrastive_loss + itm_loss_weight * itm_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # Validation
        model.eval()
        total_val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                pixel_values = batch["pixel_values"].to(device)

                image_emb, image_tokens = model.encode_image(pixel_values)
                text_emb, text_tokens = model.encode_text(input_ids, attention_mask)

                B = image_emb.size(0)
                coarse_scores = image_emb @ text_emb.T
                sim = coarse_scores / model.temperature
                contrastive_labels = torch.arange(B, device=device)
                contrastive_loss = (F.cross_entropy(sim, contrastive_labels) + F.cross_entropy(sim.T, contrastive_labels)) / 2
                itm_logits, itm_labels = build_itm_batch(image_tokens, text_tokens, attention_mask, coarse_scores)
                itm_loss = itm_criterion(itm_logits, itm_labels)
                loss = contrastive_loss + itm_loss_weight * itm_loss

                total_val_loss += loss.item()

                preds = (torch.sigmoid(itm_logits) > 0.5).float()
                correct += (preds == itm_labels).sum().item()
                total += itm_labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = correct / total

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val ITM Acc: {val_acc:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch + 1
            patience_counter = 0
            if best_weights_path is not None:
                torch.save(model.state_dict(), best_weights_path)
                print(f"Saved best checkpoint to {best_weights_path}")
        else:
            patience_counter += 1
            print(f"No val improvement. Early stopping patience: {patience_counter}/{early_stopping_patience}")
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping triggered. Best epoch: {best_epoch}, Best Val Loss: {best_val_loss:.4f}")
                break

    if best_epoch != -1:
        print(f"Training finished. Best epoch: {best_epoch}, Best Val Loss: {best_val_loss:.4f}")
    if best_weights_path is not None and os.path.exists(best_weights_path):
        model.load_state_dict(torch.load(best_weights_path, map_location=device, weights_only=True))
        print(f"Reloaded best checkpoint from {best_weights_path}")

In [9]:
train(
    model,
    batch_size=TRAIN_BATCH_SIZE,
    epochs=TRAIN_EPOCHS,
    lr=TRAIN_LR,
    hard_negative_top_k=HARD_NEGATIVE_TOP_K,
    itm_loss_weight=ITM_LOSS_WEIGHT,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    best_weights_path=BEST_LOCAL_WEIGHTS_PATH,
)

Epoch 1/2 [Val]: 100%|██████████| 159/159 [00:16<00:00,  9.81it/s]


Epoch 1/2 - Train Loss: 0.6455, Val Loss: 2.8955, Val ITM Acc: 0.4009
Saved best checkpoint to weights/best_advanced_model_bs32_e2_lr3e-5_hn3_itm0.5.pt


Epoch 2/2 [Val]: 100%|██████████| 159/159 [00:16<00:00,  9.80it/s]


Epoch 2/2 - Train Loss: 0.2587, Val Loss: 3.1144, Val ITM Acc: 0.3957
No val improvement. Early stopping patience: 1/1
Early stopping triggered. Best epoch: 1, Best Val Loss: 2.8955
Training finished. Best epoch: 1, Best Val Loss: 2.8955
Reloaded best checkpoint from weights/best_advanced_model_bs32_e2_lr3e-5_hn3_itm0.5.pt


# Save/Load Model

In [10]:
# Optional local load if you already saved a checkpoint for this experiment.
# model.load_state_dict(torch.load(LOCAL_WEIGHTS_PATH, weights_only=True))

In [11]:
from huggingface_hub import notebook_login
notebook_login()


In [13]:
os.makedirs(os.path.dirname(LOCAL_WEIGHTS_PATH), exist_ok=True)
torch.save(model.state_dict(), LOCAL_WEIGHTS_PATH)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
api.upload_file(
    path_or_fileobj=LOCAL_WEIGHTS_PATH,
    path_in_repo=HF_FILENAME,
    repo_id=HF_REPO_ID,
    repo_type="model",
)

print(f"Uploaded {LOCAL_WEIGHTS_PATH} -> {HF_REPO_ID}/{HF_FILENAME}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...2_e2_lr3e-5_hn3_itm0.5.pt:   0%|          |  567kB /  814MB            

Uploaded weights/advanced_model_bs32_e2_lr3e-5_hn3_itm0.5.pt -> Zoe3324/advanced_3/advanced_model_bs32_e2_lr3e-5_hn3_itm0.5.pt


In [12]:
# Download weights from Hugging Face Hub and load them into the current model.
ckpt_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=HF_FILENAME,
    repo_type="model",
)

model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
print(f"Loaded checkpoint from {HF_REPO_ID}/{HF_FILENAME}")

advanced_model_bs32_e2_lr3e-5_hn5.pt:   0%|          | 0.00/814M [00:00<?, ?B/s]

Loaded checkpoint from Zoe3324/advanced_3/advanced_model_bs32_e2_lr3e-5_hn5.pt


# Evaluation: 1to1

In [13]:
def eval_collate_fn(batch):
    """Collate without negatives — just return raw images and captions."""
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]
    return texts, images

In [14]:
def evaluate_image_to_text_1to1(model, data, num_samples=200, top_k=20):
    """Two-stage retrieval: coarse dual-encoder retrieval → cross-attention re-ranking."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_fn)

    all_texts = []
    all_images = []
    for texts, images in loader:
        all_texts.extend(texts)
        all_images.extend(images)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        # Encode all images
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        # Encode all texts
        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)  # (N, embed_dim)
    all_text_embs = torch.cat(all_text_embs)    # (N, embed_dim)

    # Coarse similarity matrix
    coarse_sim = all_image_embs @ all_text_embs.T  # (N, N)

    # ── Stage 2: Re-rank top-k candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)  # only fill top-k entries

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking"):
            # Get top-k caption candidates for image_i
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            # Prepare image_i paired with each candidate caption
            img = all_images[i]
            cand_texts = [all_texts[j] for j in candidates]
            image_enc = image_processor(images=[img] * top_k, return_tensors="pt")
            text_enc = tokenizer(cand_texts, padding="max_length", truncation=True, max_length=32, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
                text_enc["attention_mask"].to(device),
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k):
        correct = 0
        for i in range(score_mat.size(0)):
            topk = score_mat[i].topk(k).indices
            if i in topk:
                correct += 1
        return correct / score_mat.size(0)

    # Image-to-text
    print()
    print(f"=== Stage 1 Only: Coarse Retrieval (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10):.4f}")

    print(f"\n=== Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10):.4f}")


In [16]:
def evaluate_text_to_image_1to1(model, data, num_samples=200, top_k=20):
    """Two-stage TEXT-TO-IMAGE retrieval (ungrouped, 1-to-1 matching assumed)."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_fn)

    all_texts = []
    all_images = []
    for texts, images in loader:
        all_texts.extend(texts)
        all_images.extend(images)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)

    # Text-to-image: each row is a query text, columns are images
    coarse_sim = all_text_embs @ all_image_embs.T  # (N, N)

    # ── Stage 2: Re-rank top-k image candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking (T→I)"):
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            text = all_texts[i]
            cand_images = [all_images[j] for j in candidates]
            text_enc = tokenizer([text] * top_k, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            image_enc = image_processor(images=cand_images, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
                text_enc["attention_mask"].to(device),
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k):
        correct = 0
        for i in range(score_mat.size(0)):
            topk = score_mat[i].topk(k).indices
            if i in topk:
                correct += 1
        return correct / score_mat.size(0)

    print()
    print(f"=== [Text→Image] Stage 1 Only: Coarse Retrieval (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10):.4f}")

    print(f"\n=== [Text→Image] Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10):.4f}")


# Evaluation: grouped

In [17]:
def eval_collate_grouped(batch):
    """Collate that also returns img_id for grouping."""
    texts = [x["caption"] for x in batch]
    images = [x["image"].convert("RGB") for x in batch]
    img_ids = [x["img_id"] for x in batch]
    return texts, images, img_ids

In [18]:
def evaluate_image_to_text_grouped(model, data, num_samples=200, top_k=20):
    """Two-stage retrieval with grouped recall (all captions for same image are correct)."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_grouped)

    all_texts = []
    all_images = []
    all_img_ids = []
    for texts, images, img_ids in loader:
        all_texts.extend(texts)
        all_images.extend(images)
        all_img_ids.extend(img_ids)

    # Build mapping: img_id -> list of indices
    group_to_indices = {}
    for idx, gid in enumerate(all_img_ids):
        if gid not in group_to_indices:
            group_to_indices[gid] = []
        group_to_indices[gid].append(idx)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)

    coarse_sim = all_image_embs @ all_text_embs.T

    # ── Stage 2: Re-rank top-k candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking"):
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            img = all_images[i]
            cand_texts = [all_texts[j] for j in candidates]
            image_enc = image_processor(images=[img] * top_k, return_tensors="pt")
            text_enc = tokenizer(cand_texts, padding="max_length", truncation=True, max_length=32, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
                text_enc["attention_mask"].to(device),
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k, idx_to_group, group_to_idx):
        total_recall = 0.0
        for i in range(score_mat.size(0)):
            topk = set(score_mat[i].topk(k).indices.tolist())
            correct_set = set(group_to_idx[idx_to_group[i]])
            n_relevant = len(correct_set)
            n_retrieved = len(topk & correct_set)
            total_recall += n_retrieved / n_relevant
        return total_recall / score_mat.size(0)

    print()
    print(f"=== Stage 1 Only: Coarse Retrieval (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10, all_img_ids, group_to_indices):.4f}")

    print(f"\n=== Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10, all_img_ids, group_to_indices):.4f}")


In [19]:
def evaluate_text_to_image_grouped(model, data, num_samples=200, top_k=20):
    """Two-stage TEXT-TO-IMAGE retrieval with grouped recall."""
    subset = data.select(range(min(num_samples, len(data))))
    loader = DataLoader(subset, batch_size=32, shuffle=False, collate_fn=eval_collate_grouped)

    all_texts = []
    all_images = []
    all_img_ids = []
    for texts, images, img_ids in loader:
        all_texts.extend(texts)
        all_images.extend(images)
        all_img_ids.extend(img_ids)

    # Build mapping: img_id -> list of indices
    group_to_indices = {}
    for idx, gid in enumerate(all_img_ids):
        if gid not in group_to_indices:
            group_to_indices[gid] = []
        group_to_indices[gid].append(idx)

    N = len(all_texts)

    # ── Stage 1: Encode all images and texts independently ──
    print("Stage 1: Computing embeddings...")
    all_image_embs = []
    all_text_embs = []
    model.eval()
    bs = 64
    with torch.no_grad():
        for start in range(0, N, bs):
            end = min(start + bs, N)
            image_enc = image_processor(images=all_images[start:end], return_tensors="pt")
            img_emb, _ = model.encode_image(image_enc["pixel_values"].to(device))
            all_image_embs.append(img_emb.cpu())

        for start in range(0, N, bs):
            end = min(start + bs, N)
            text_enc = tokenizer(all_texts[start:end], padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            txt_emb, _ = model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))
            all_text_embs.append(txt_emb.cpu())

    all_image_embs = torch.cat(all_image_embs)
    all_text_embs = torch.cat(all_text_embs)

    # Text-to-image coarse similarity: each row is a query text, columns are images
    coarse_sim = all_text_embs @ all_image_embs.T  # (N, N)

    # ── Stage 2: Re-rank top-k image candidates with cross-attention ──
    print(f"Stage 2: Re-ranking top-{top_k} candidates...")
    score_matrix = torch.full((N, N), -1.0)

    with torch.no_grad():
        for i in tqdm(range(N), desc="Re-ranking (T→I)"):
            # For text_i, get top-k image candidates
            candidates = coarse_sim[i].topk(top_k).indices.tolist()

            text = all_texts[i]
            cand_images = [all_images[j] for j in candidates]
            text_enc = tokenizer([text] * top_k, padding="max_length", truncation=True, max_length=32, return_tensors="pt")
            image_enc = image_processor(images=cand_images, return_tensors="pt")

            itm_logits = model.compute_itm(
                model.encode_text(text_enc["input_ids"].to(device), text_enc["attention_mask"].to(device))[1],
                model.encode_image(image_enc["pixel_values"].to(device))[1],
                text_enc["attention_mask"].to(device),
            )
            itm_scores = torch.sigmoid(itm_logits).cpu()

            for rank, j in enumerate(candidates):
                score_matrix[i][j] = itm_scores[rank]

    def recall_at_k(score_mat, k, idx_to_group, group_to_idx):
        total_recall = 0.0
        for i in range(score_mat.size(0)):
            topk = set(score_mat[i].topk(k).indices.tolist())
            correct_set = set(group_to_idx[idx_to_group[i]])
            n_relevant = len(correct_set)
            n_retrieved = len(topk & correct_set)
            total_recall += n_retrieved / n_relevant
        return total_recall / score_mat.size(0)

    print()
    print(f"=== [Text→Image] Stage 1 Only: Coarse Retrieval (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(coarse_sim, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(coarse_sim, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(coarse_sim, 10, all_img_ids, group_to_indices):.4f}")

    print(f"\n=== [Text→Image] Two-Stage: Coarse + Re-Rank top-{top_k} (N={N}, grouped) ===")
    print(f"  Recall@1:  {recall_at_k(score_matrix, 1, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@5:  {recall_at_k(score_matrix, 5, all_img_ids, group_to_indices):.4f}")
    print(f"  Recall@10: {recall_at_k(score_matrix, 10, all_img_ids, group_to_indices):.4f}")


# Run Evaluation

In [20]:
evaluate_image_to_text_1to1(model, test_data, num_samples=EVAL_NUM_SAMPLES, top_k=EVAL_TOP_K)
evaluate_text_to_image_1to1(model, test_data, num_samples=EVAL_NUM_SAMPLES, top_k=EVAL_TOP_K)
evaluate_image_to_text_grouped(model, test_data, num_samples=EVAL_NUM_SAMPLES, top_k=EVAL_TOP_K)
evaluate_text_to_image_grouped(model, test_data, num_samples=EVAL_NUM_SAMPLES, top_k=EVAL_TOP_K)

Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking: 100%|██████████| 1000/1000 [02:08<00:00,  7.79it/s]



=== Stage 1 Only: Coarse Retrieval (N=1000) ===
  Recall@1:  0.1620
  Recall@5:  0.6550
  Recall@10: 0.8290

=== Two-Stage: Coarse + Re-Rank top-20 (N=1000) ===
  Recall@1:  0.1520
  Recall@5:  0.6420
  Recall@10: 0.8100
Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking (T→I): 100%|██████████| 1000/1000 [02:10<00:00,  7.67it/s]



=== [Text→Image] Stage 1 Only: Coarse Retrieval (N=1000) ===
  Recall@1:  0.1500
  Recall@5:  0.7060
  Recall@10: 0.8340

=== [Text→Image] Two-Stage: Coarse + Re-Rank top-20 (N=1000) ===
  Recall@1:  0.1500
  Recall@5:  0.6930
  Recall@10: 0.8220
Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking: 100%|██████████| 1000/1000 [02:09<00:00,  7.71it/s]



=== Stage 1 Only: Coarse Retrieval (N=1000, grouped) ===
  Recall@1:  0.1620
  Recall@5:  0.6550
  Recall@10: 0.8290

=== Two-Stage: Coarse + Re-Rank top-20 (N=1000, grouped) ===
  Recall@1:  0.1520
  Recall@5:  0.6420
  Recall@10: 0.8100
Stage 1: Computing embeddings...
Stage 2: Re-ranking top-20 candidates...


Re-ranking (T→I): 100%|██████████| 1000/1000 [02:10<00:00,  7.66it/s]



=== [Text→Image] Stage 1 Only: Coarse Retrieval (N=1000, grouped) ===
  Recall@1:  0.1412
  Recall@5:  0.7060
  Recall@10: 0.8340

=== [Text→Image] Two-Stage: Coarse + Re-Rank top-20 (N=1000, grouped) ===
  Recall@1:  0.1386
  Recall@5:  0.6930
  Recall@10: 0.8220
